In [1]:
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model
from datetime import timedelta, datetime

In [3]:
# ==========================================
# 1. SETTINGS
# ==========================================
TARGET_DATE = "2025-01-15"  # <--- CHANGE THIS DATE to test different futures
STATION_NAME = "Peenya"     # <--- The station you want to check

model = load_model('aqi_cnn_lstm_model.h5')
scaler = joblib.load('aqi_scaler.pkl')

# Load original data to get the "Last 60 Days"
df = pd.read_csv('C:\\Users\\Chinmaya holla\\Desktop\\Airpollution\\Data\\Processed_data\\master_cleaned_data.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'], dayfirst=True)
df_pivot = df.pivot_table(index='Timestamp', columns='Station', values='PM2_5')
df_pivot = df_pivot.interpolate(method='time').resample('D').mean().bfill().ffill()

# Get the list of stations to map predictions back to names
stations = df_pivot.columns.tolist()
print(f"Stations: {stations}")

Stations: ['BTM', 'BapujiNagar', 'Hebbal', 'Hombegowda', 'Jayanagar', 'Kadabasenahalli', 'Peenya', 'RVCE', 'Silkboard']


In [4]:
# ==========================================
# 3. PREPARE THE STARTING WINDOW
# ==========================================
# We start predicting from the very next day after your dataset ends
last_date = df_pivot.index[-1]
print(f"Dataset ends on: {last_date.date()}")

# Get the last 60 days of data (scaled)
last_60_days = df_pivot.values[-60:]
current_batch = scaler.transform(last_60_days)  # Scale it to 0-1
current_batch = current_batch.reshape(1, 60, len(stations)) # Reshape for Model

Dataset ends on: 2024-12-31


c:\Users\Chinmaya holla\anaconda3\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [5]:
# ==========================================
# 4. RECURSIVE PREDICTION LOOP
# ==========================================
target_date_obj = datetime.strptime(TARGET_DATE, "%Y-%m-%d")
days_to_predict = (target_date_obj - last_date).days

if days_to_predict <= 0:
    print("Error: Target date must be in the future!")
    exit()

print(f"Predicting {days_to_predict} days into the future... (This may take a moment)")

future_predictions = []
current_date = last_date

for i in range(days_to_predict):
    # 1. Predict the next day
    next_day_scaled = model.predict(current_batch, verbose=0)
    
    # 2. Store the prediction
    future_predictions.append(next_day_scaled[0])
    
    # 3. Update the batch: Drop Day 0, Add Predicted Day at the end
    # Reshape next_day_scaled to (1, 1, n_stations) to stack it
    next_input = next_day_scaled.reshape(1, 1, len(stations))
    current_batch = np.append(current_batch[:, 1:, :], next_input, axis=1)
    
    current_date += timedelta(days=1)

Predicting 15 days into the future... (This may take a moment)


In [6]:
# ==========================================
# 5. DECODE AND DISPLAY RESULTS
# ==========================================
# Inverse transform to get actual PM2.5 values
future_predictions = scaler.inverse_transform(future_predictions)

# Find the prediction for the exact target date
# The last entry in the list is the target date
final_prediction = future_predictions[-1]

print(f"\n{'='*40}")
print(f"FORECAST FOR {TARGET_DATE}")
print(f"{'='*40}")

# Show specific station result
if STATION_NAME in stations:
    station_idx = stations.index(STATION_NAME)
    pm25_value = final_prediction[station_idx]
    
    # Policy Logic
    status = "Good"
    if pm25_value > 60: status = "Satisfactory"
    if pm25_value > 90: status = "Moderately Polluted"
    if pm25_value > 120: status = "Poor"
    if pm25_value > 250: status = "Severe"
    
    print(f"Station: {STATION_NAME}")
    print(f"Predicted PM2.5: {pm25_value:.2f}")
    print(f"Status: {status}")
else:
    print(f"Station '{STATION_NAME}' not found.")

print("\nAll Stations:")
for i, station in enumerate(stations):
    print(f" - {station}: {final_prediction[i]:.2f}")


FORECAST FOR 2025-01-15
Station: Peenya
Predicted PM2.5: 44.10
Status: Good

All Stations:
 - BTM: 33.91
 - BapujiNagar: 53.94
 - Hebbal: 38.13
 - Hombegowda: 39.17
 - Jayanagar: 53.68
 - Kadabasenahalli: 31.75
 - Peenya: 44.10
 - RVCE: 23.02
 - Silkboard: 46.04
